# **Scraping Amazon.Com**

In [7]:
import argparse
import csv
import os
import random
import time
import urllib.parse
from datetime import datetime, timezone
from pathlib import Path

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

In [8]:
try:
    from webdriver_manager.chrome import ChromeDriverManager
except Exception:
    ChromeDriverManager = None

def build_driver(headless=True):
    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument(
        "--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
    )

    if ChromeDriverManager is not None:
        service = Service(ChromeDriverManager().install())
        return webdriver.Chrome(service=service, options=options)

    return webdriver.Chrome(options=options)

In [9]:
def parse_results(page_html, base_url="https://www.amazon.com"):
    soup = BeautifulSoup(page_html, "html.parser")
    results = []

    for item in soup.select("div.s-main-slot div[data-asin]"):
        asin = item.get("data-asin")
        if not asin:
            continue

        title_tag = item.select_one("h2 a.a-link-normal span")
        link_tag = item.select_one("h2 a.a-link-normal")
        title = title_tag.get_text(strip=True) if title_tag else ""
        link = ""
        if link_tag and link_tag.get("href"):
            href = link_tag.get("href")
            link = href if href.startswith("http") else urllib.parse.urljoin(base_url, href)

        price_whole = ""
        price_fraction = ""
        whole_tag = item.select_one("span.a-price-whole")
        frac_tag = item.select_one("span.a-price-fraction")
        if whole_tag:
            price_whole = whole_tag.get_text(strip=True).replace(",", "")
        if frac_tag:
            price_fraction = frac_tag.get_text(strip=True)

        rating = ""
        num_ratings = ""
        rating_tag = item.select_one("span.a-icon-alt")
        rating_count_tag = item.select_one(".a-row.a-size-small span.a-size-base")
        if rating_tag:
            rating = rating_tag.get_text(strip=True)
        if rating_count_tag:
            num_ratings = rating_count_tag.get_text(strip=True)

        results.append(
            {
                "asin": asin,
                "title": title,
                "link": link,
                "price_whole": price_whole,
                "price_fraction": price_fraction,
                "rating": rating,
                "num_ratings": num_ratings,
            }
        )

    return results


In [10]:
def save_csv(rows, out_path):
    fieldnames = [
        "asin",
        "title",
        "link",
        "price_whole",
        "price_fraction",
        "rating",
        "num_ratings",
    ]
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

In [11]:
def run(keyword, pages=3, headless=True, out_dir="outputs"):
    driver = build_driver(headless=headless)
    all_rows = []
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    safe_kw = urllib.parse.quote_plus(keyword)

    try:
        for page in range(1, pages + 1):
            url = f"https://www.amazon.com/s?k={urllib.parse.quote(keyword)}&page={page}"
            print(f"Loading page {page}: {url}")
            driver.get(url)

            try:
                WebDriverWait(driver, 12).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "div.s-main-slot"))
                )
            except Exception:
                print("Warning: results container didn't appear within timeout")

            page_html = driver.page_source
            os.makedirs(out_dir, exist_ok=True)

            html_path = os.path.join(out_dir, f"{safe_kw}_page{page}_{timestamp}.html")
            with open(html_path, "w", encoding="utf-8") as html_file:
                html_file.write(page_html)
            print(f"Saved HTML to {html_path}")

            rows = parse_results(page_html)
            print(f"Extracted {len(rows)} items from page {page}")
            all_rows.extend(rows)

            time.sleep(random.uniform(1.0, 3.0))
    finally:
        driver.quit()

    csv_path = os.path.join(out_dir, f"{safe_kw}_results_{timestamp}.csv")
    save_csv(all_rows, csv_path)
    print(f"Saved CSV to {csv_path}")
    return csv_path

In [12]:
keyword = "wireless earbuds"
pages = 3
output_csv = run(keyword=keyword, pages=pages, headless=True, out_dir="outputs")
print(f"Saved CSV: {output_csv}")

Loading page 1: https://www.amazon.com/s?k=wireless%20earbuds&page=1
Saved HTML to outputs/wireless+earbuds_page1_20260507_101545.html
Extracted 130 items from page 1
Loading page 2: https://www.amazon.com/s?k=wireless%20earbuds&page=2
Saved HTML to outputs/wireless+earbuds_page2_20260507_101545.html
Extracted 130 items from page 2
Loading page 3: https://www.amazon.com/s?k=wireless%20earbuds&page=3
Saved HTML to outputs/wireless+earbuds_page3_20260507_101545.html
Extracted 130 items from page 3
Saved CSV to outputs/wireless+earbuds_results_20260507_101545.csv
Saved CSV: outputs/wireless+earbuds_results_20260507_101545.csv
